# Conservative 2D regrid — unstructured mesh + save/load

Unstructured meshes (ICON triangles, MPAS hexagons, finite-element
models) are never 1D-separable. `ConservativeRegridder.from_polygons`
takes any flat array of shapely polygons as source or target.

This notebook builds a synthetic Voronoi hex-like mesh and regrids a
structured source onto it, then persists the regridder so a
long-running pipeline can skip the weight build on restart.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from scipy.spatial import Voronoi
import shapely

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder, polygons_from_coords

## Build a Voronoi mesh

Jittered grid points → Voronoi → clip to the region of interest. Real
pipelines load a pre-built mesh (UGRID, ICON, etc.); the point here is
that `from_polygons` needs only a 1D array of shapely polygons.

In [ ]:
def voronoi_mesh(n_points, bbox, seed=0):
    rng = np.random.default_rng(seed)
    x0, y0, x1, y1 = bbox
    side = int(np.sqrt(n_points))
    xs, ys = np.linspace(x0, x1, side), np.linspace(y0, y1, side)
    pts = np.column_stack([np.repeat(xs, side), np.tile(ys, side)])
    pts += rng.normal(scale=(x1 - x0) / side * 0.25, size=pts.shape)
    halo = np.array([
        [2*x0 - x1, 2*y0 - y1], [2*x1 - x0, 2*y0 - y1],
        [2*x0 - x1, 2*y1 - y0], [2*x1 - x0, 2*y1 - y0],
    ])
    vor = Voronoi(np.concatenate([pts, halo]))
    clip = shapely.box(x0, y0, x1, y1)
    polys = []
    for i in range(len(pts)):
        r = vor.regions[vor.point_region[i]]
        if not r or -1 in r:
            continue
        p = shapely.intersection(shapely.Polygon(vor.vertices[r]), clip)
        if p.is_empty or p.geom_type != "Polygon":
            continue
        polys.append(p)
    return np.array(polys, dtype=object)

bbox = (-120, -50, 120, 50)
mesh_polys = voronoi_mesh(n_points=400, bbox=bbox)
print(f"{len(mesh_polys)} mesh cells")

## Structured lat/lon source

In [ ]:
lat_s = np.linspace(-50, 50, 100, endpoint=False) + 0.5
lon_s = np.linspace(-120, 120, 240, endpoint=False) + 0.5
Lo, La = np.meshgrid(lon_s, lat_s)
src = xr.DataArray(
    np.sin(np.deg2rad(Lo) * 2) * np.cos(np.deg2rad(La) * 3),
    dims=("latitude", "longitude"),
    coords={"latitude": lat_s, "longitude": lon_s},
)

## Regrid onto the mesh

In [ ]:
rgr = ConservativeRegridder.from_polygons(
    source_polygons=polygons_from_coords(lon_s, lat_s),
    target_polygons=mesh_polys,
    source_dim="src_cell",
    target_dim="cell",
)
print(rgr)

src_flat = xr.DataArray(src.values.ravel(), dims=("src_cell",))
mesh_vals = rgr.regrid(src_flat)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
patches = [np.asarray(p.exterior.coords) for p in mesh_polys]
pc = PolyCollection(patches, array=mesh_vals.values, cmap="RdBu_r",
                    edgecolor="0.4", lw=0.3, clim=(-1, 1))
ax.add_collection(pc)
ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])
ax.set_aspect("equal")
fig.colorbar(pc, ax=ax, shrink=0.8)
ax.set_title(f"regridded onto {len(mesh_polys)}-cell Voronoi mesh")

## Persist the regridder

For a fixed source/target pair, the weight matrix is the same forever.
Saving it lets long-running pipelines skip the (expensive) build on
restart.

In [ ]:
path = Path(tempfile.gettempdir()) / "mesh_regridder.nc"
rgr.to_netcdf(path)

with xr.open_dataset(path) as weights:
    for k in ("xarray_regrid_version", "created", "src_shape", "dst_shape"):
        print(f"  {k}: {weights.attrs[k]}")

rgr2 = ConservativeRegridder.from_netcdf(path)
same = np.array_equal(rgr.regrid(src_flat).values, rgr2.regrid(src_flat).values)
print(f"\nreload bit-identical: {same}")